In [0]:
INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'STATUS_CHANGED', cast(current_timestamp() as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  current_timestamp() AS event_ts,
  'NOTE_ADDED' AS event_type,
  :p_event_user AS event_user,
  NULL AS old_status,
  NULL AS new_status,
  NULL AS old_priority,
  NULL AS new_priority,
  NULL AS old_owner,
  NULL AS new_owner,
  NULL AS old_due_date,
  NULL AS new_due_date,
  :p_note_text AS note_text,
  'Issue note added' AS event_comment
FROM cfpb_risk.app.issues i
WHERE i.issue_id = :p_issue_id
AND coalesce(trim(:p_note_text), '') <> '';

UPDATE cfpb_risk.app.issues
SET
  latest_note = :p_note_text,
  updated_ts = current_timestamp()
WHERE issue_id = :p_issue_id
AND coalesce(trim(:p_note_text), '') <> '';